# Prompt Templates & Output Parsers
### Build reusable, dynamic prompts and parse structured outputs

---
**Topics Covered:**
- `PromptTemplate` for single-variable and multi-variable prompts
- `ChatPromptTemplate` with system + user roles
- `FewShotChatMessagePromptTemplate` for in-context examples
- `StrOutputParser` — strip `AIMessage` wrapper to plain text
- `JsonOutputParser` — force structured JSON responses
- Composing prompts into a simple LCEL pipeline with `|`


In [ ]:
!pip install langchain langchain-groq langchain-google-genai python-dotenv

In [ ]:
pip install --upgrade pip

In [ ]:
pip install langchain-groq

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model

llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq", temperature=0.6)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


## 1. `PromptTemplate` — Simple String Templates

Use curly-brace placeholders `{variable}` to inject values at call time.

In [2]:
from langchain_core.prompts import PromptTemplate

explain_template = PromptTemplate(
    input_variables=["concept", "audience"],
    template="Explain {concept} to a {audience} in no more than 100 words.",
)

# Render the template — no LLM call yet
rendered = explain_template.format(concept="neural networks", audience="10-year-old")
print(rendered)

Explain neural networks to a 10-year-old in no more than 100 words.


In [3]:
# Now invoke the LLM with the rendered prompt
response = llm.invoke(rendered)
print(response.content)

Imagine your brain has lots of tiny helpers that talk to each other. A neural network is like a computer version of that. It's made up of many tiny "helpers" (called neurons) that work together to learn and solve problems. They look at information, talk to each other, and figure out answers, like recognizing pictures or understanding what you say. The more they practice, the smarter they get!


## 2. `ChatPromptTemplate` — Role-Aware Templates

Use `("system", ...)` and `("human", ...)` tuples to assign message roles declaratively.

In [4]:
from langchain_core.prompts import ChatPromptTemplate

blog_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical blog writer who writes engaging, SEO-friendly posts."),
    ("human", "Write a 150-word intro paragraph for a blog post titled: '{title}'.")
])

# Format and inspect before calling the model
messages = blog_prompt.format_messages(title="Why Python Type Hints Will Save Your Sanity")
for m in messages:
    print(f"[{m.type.upper()}]:", m.content[:80], "...")

[SYSTEM]: You are a technical blog writer who writes engaging, SEO-friendly posts. ...
[HUMAN]: Write a 150-word intro paragraph for a blog post titled: 'Why Python Type Hints  ...


In [5]:
result = llm.invoke(messages)
print(result.content)

As Python developers, we've all been there - pouring over lines of code, trying to decipher the intent behind a particular function or variable. Without explicit type definitions, it's easy to get lost in a sea of ambiguity, leading to frustrating debugging sessions and wasted hours. But what if you could avoid this chaos altogether? Enter Python type hints, a game-changing feature that allows you to add optional type annotations to your code. Introduced in Python 3.5, type hints have been gaining traction as a best practice in the Python community. By incorporating type hints into your workflow, you can significantly improve code readability, reduce errors, and enhance maintainability. In this post, we'll explore the benefits of Python type hints and show you how they can be a sanity-saving addition to your development toolkit. With type hints, you can write better code, faster.


## 3. `StrOutputParser` — Clean String Output

Chain a parser with `|` to strip the `AIMessage` envelope and return a plain `str`.

In [6]:
from langchain_core.output_parsers import StrOutputParser

# Build a minimal pipeline: template → llm → parser
haiku_prompt = ChatPromptTemplate.from_messages([
    ("system", "You only reply with haiku poetry. Strictly 5-7-5 syllables."),
    ("human", "Write a haiku about {topic}.")
])

chain = haiku_prompt | llm | StrOutputParser()

# chain.invoke returns a plain string now
poem = chain.invoke({"topic": "machine learning overfitting"})
print(type(poem))   # <class 'str'>
print(poem)

<class 'langchain_core.messages.base.TextAccessor'>
Data fits too well
Noise and truth in equal parts
Model's gentle lie


## 4. `JsonOutputParser` — Structured Data Extraction

Ask the model to return JSON and automatically parse it into a Python dict.

In [7]:
from langchain_core.output_parsers import JsonOutputParser

json_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You respond ONLY with valid JSON. No markdown fences, no extra text.\n"
        "Schema: {{\"name\": str, \"founded\": int, \"hq\": str, \"employees_k\": float}}"
    )),
    ("human", "Give me the company profile for {company}.")
])

json_chain = json_prompt | llm | JsonOutputParser()

profile = json_chain.invoke({"company": "DeepMind"})
print(type(profile))   # <class 'dict'>
print(profile)

<class 'dict'>
{'name': 'DeepMind', 'founded': 2010, 'hq': 'London', 'employees_k': 1000.0}


In [8]:
# Access fields directly like a normal Python dict
print("Company  :", profile["name"])
print("Founded  :", profile["founded"])
print("HQ       :", profile["hq"])

Company  : DeepMind
Founded  : 2010
HQ       : London


## 5. Few-Shot Prompting with `ChatPromptTemplate`

Provide worked examples inside the prompt to steer the model toward a specific output format.

In [9]:
# Manual few-shot via list of messages
few_shot_messages = [
    ("system", "You classify the sentiment of a code review comment as POSITIVE, NEGATIVE, or NEUTRAL."),
    ("human",  "Comment: 'This function is clean and well-named.'"),
    ("ai",     "POSITIVE"),
    ("human",  "Comment: 'Magic numbers everywhere — please use named constants.'"),
    ("ai",     "NEGATIVE"),
    ("human",  "Comment: 'Added unit test for the edge case.'"),
    ("ai",     "POSITIVE"),
    ("human",  "Comment: '{comment}'"),
]

classifier_prompt = ChatPromptTemplate.from_messages(few_shot_messages)
classifier_chain  = classifier_prompt | llm | StrOutputParser()

samples = [
    "Indentation is inconsistent throughout the file.",
    "Good use of list comprehension here.",
    "I've reviewed this PR.",
]

for sample in samples:
    label = classifier_chain.invoke({"comment": sample})
    print(f"{label:10s} → {sample}")

NEGATIVE   → Indentation is inconsistent throughout the file.
POSITIVE   → Good use of list comprehension here.
NEUTRAL    → I've reviewed this PR.


## 6. Multi-Variable Chat Template — Code Review Bot

In [10]:
review_prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are a senior {language} engineer. Review the code snippet below.\n"
        "Focus on: correctness, readability, and performance. Keep your review under 120 words."
    )),
    ("human", "```{language}\n{code}\n```")
])

review_chain = review_prompt | llm | StrOutputParser()

code_snippet = """
def find_duplicates(lst):
    duplicates = []
    for i in range(len(lst)):
        for j in range(i + 1, len(lst)):
            if lst[i] == lst[j] and lst[i] not in duplicates:
                duplicates.append(lst[i])
    return duplicates
"""

feedback = review_chain.invoke({"language": "Python", "code": code_snippet})
print(feedback)

**Code Review**

The code is correct but inefficient (O(n^2) complexity). Readability is fair. For better performance, consider using a set or dictionary to track duplicates, reducing complexity to O(n). Example: `seen = set(); duplicates = set(); for item in lst: if item in seen: duplicates.add(item); seen.add(item)`.


---
### Summary

| Class | Purpose |
|-------|---------|
| `PromptTemplate` | Single-string template with `{vars}` |
| `ChatPromptTemplate` | Role-structured template `("system",...), ("human",...)`|
| `StrOutputParser` | `AIMessage` → `str` |
| `JsonOutputParser` | `AIMessage` → `dict` |
| `chain = prompt \| llm \| parser` | LCEL pipeline |

**Next → `03_tool_calling.ipynb`**